# Image Genration Using GAN Demonstration In Python

## Background

Image datasets such as CIFAR-10 dataset are widely used for training computer vision models. However, in real-world scenarios, collecting large volumes of labeled images is often expensive, time-consuming, and sometimes impractical.

Many applications also suffer from data imbalance, where certain classes have significantly fewer examples (e.g., rare objects, defects, or edge cases). This leads to biased models that perform poorly on underrepresented categories.

Additionally, domains like healthcare, manufacturing defects, and surveillance often deal with rare or hard-to-capture events, making it difficult to build robust datasets through traditional data collection methods alone.

To address these challenges, synthetic image generation using models like GANs (Generative Adversarial Networks) is increasingly used to augment datasets, improve diversity, and reduce dependency on manual data collection.

## Objective
The objective of this case study is to generate additional images using the CIFAR-10 dataset to:

Enhance dataset size without additional data collection cost
Address class imbalance by generating samples for underrepresented classes
Improve model generalization by increasing data variability
Demonstrate the use of generative models (e.g., GANs) for data augmentation
Evaluate whether synthetic images can improve downstream model performance ( broader scope , not included in this code)

### Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import save_image


### Device Settings

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nz = 100

### Generator


In [3]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        # Generator model: converts random noise (latent vector) into an image
        self.model = nn.Sequential(

            # Input: (nz x 1 x 1)
            # Step 1: Upsample to (256 x 4 x 4)
            nn.ConvTranspose2d(nz, 256, 4, 1, 0, bias=False),
            nn.BatchNorm2d(256),   # Normalize for stable training
            nn.ReLU(True),         # Activation

            # Step 2: Upsample to (128 x 8 x 8)
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            # Step 3: Upsample to (64 x 16 x 16)
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            # Step 4: Upsample to (3 x 32 x 32) → Final RGB image
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh()  # Output in range [-1, 1]
        )

    def forward(self, x):
        # Forward pass: generate image from noise
        return self.model(x)

### Discriminator


In [4]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        # Discriminator model: classifies image as Real (1) or Fake (0)
        self.model = nn.Sequential(

            # Input: (3 x 32 x 32) → RGB image
            # Step 1: Downsample to (64 x 16 x 16)
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),  # Allows small negative gradients

            # Step 2: Downsample to (128 x 8 x 8)
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),             # Stabilizes training
            nn.LeakyReLU(0.2, inplace=True),

            # Step 3: Downsample to (256 x 4 x 4)
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            # Step 4: Final output → (1 x 1 x 1)
            nn.Conv2d(256, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()  # Output probability (0 to 1)
        )

    def forward(self, x):
        # Forward pass: classify image
        return self.model(x).view(-1)  # Flatten output to (batch_size)

### Initialize models


In [5]:
netG = Generator().to(device)
netD = Discriminator().to(device)

### Load CIFAR Dataset

In [6]:
# Define image preprocessing pipeline
transform = transforms.Compose([
    transforms.Resize(32),        # Ensure all images are 32x32
    transforms.ToTensor(),        # Convert image to PyTorch tensor (0 to 1)
    transforms.Normalize((0.5, 0.5, 0.5),   # Normalize each RGB channel
                         (0.5, 0.5, 0.5))   # Final range becomes [-1, 1]
])

# Load CIFAR-10 dataset
dataset = torchvision.datasets.CIFAR10(
    root='./data',        # Directory to store dataset
    train=True,           # Use training dataset
    download=True,        # Download if not already present
    transform=transform   # Apply preprocessing
)

# Use only a subset of data for faster training (reduce training time)
subset_indices = range(20000)
dataset = torch.utils.data.Subset(dataset, subset_indices)

# Create DataLoader to load data in batches
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=64,   # Number of images per batch
    shuffle=True     # Shuffle data for better training
)

Files already downloaded and verified


### Loss & Optimizer


In [7]:
# Binary Cross Entropy Loss
# Used to measure how well discriminator distinguishes real vs fake
criterion = nn.BCELoss()

# Optimizer for Discriminator
optD = optim.Adam(
    netD.parameters(),      # Discriminator parameters
    lr=0.0002,              # Learning rate
    betas=(0.5, 0.999)      # Stabilizes GAN training
)

# Optimizer for Generator
optG = optim.Adam(
    netG.parameters(),      # Generator parameters
    lr=0.0002,
    betas=(0.5, 0.999)
)

# Fixed noise vector
# Used to generate same images during training for consistent comparison
fixed_noise = torch.randn(
    16, nz, 1, 1, device=device  # 16 samples from latent space
)

### Training


In [8]:
epochs = 30  # Number of training epochs

for epoch in range(epochs):
    for i, (real, _) in enumerate(dataloader):

        b_size = real.size(0)        # Batch size
        real = real.to(device)       # Move real images to GPU/CPU

        # Add small noise to real images (stabilization trick)
        real = real + 0.05 * torch.randn_like(real)

        # =====================
        # Train Discriminator
        # =====================
        netD.zero_grad()  # Reset gradients

        # Real labels (label smoothing → 0.9 instead of 1.0)
        label_real = torch.full((b_size,), 0.9, device=device)
        output_real = netD(real)                     # Discriminator output on real images
        loss_real = criterion(output_real, label_real)

        # Generate fake images
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake = netG(noise)

        # Fake labels (0.0)
        label_fake = torch.full((b_size,), 0.0, device=device)
        output_fake = netD(fake.detach())  # detach → no gradient to Generator
        loss_fake = criterion(output_fake, label_fake)

        # Total Discriminator loss
        loss_D = loss_real + loss_fake
        loss_D.backward()   # Backpropagation
        optD.step()         # Update Discriminator

        # =====================
        # Train Generator
        # =====================
        netG.zero_grad()  # Reset gradients

        # Generator tries to fool discriminator → labels = 1
        label_gen = torch.full((b_size,), 1.0, device=device)
        output = netD(fake)                     # Pass fake images again
        loss_G = criterion(output, label_gen)

        loss_G.backward()   # Backpropagation
        optG.step()         # Update Generator

    # Print losses after each epoch
    print(f"Epoch [{epoch+1}/{epochs}]  Loss_D: {loss_D.item():.4f}  Loss_G: {loss_G.item():.4f}")

    # Save generated images after each epoch (for tracking progress)
    with torch.no_grad():
        fake = netG(fixed_noise)                   # Generate fixed samples
        save_image((fake + 1) / 2,                # Rescale from [-1,1] → [0,1]
                   f"epoch_{epoch+1}.png",
                   nrow=4)                        # Save as grid (4x4)

Epoch [1/30]  Loss_D: 0.8935  Loss_G: 2.8350
Epoch [2/30]  Loss_D: 0.7649  Loss_G: 2.3195
Epoch [3/30]  Loss_D: 0.7940  Loss_G: 3.5487
Epoch [4/30]  Loss_D: 1.0321  Loss_G: 1.1597
Epoch [5/30]  Loss_D: 0.7091  Loss_G: 1.9095
Epoch [6/30]  Loss_D: 1.3113  Loss_G: 1.7556
Epoch [7/30]  Loss_D: 0.8842  Loss_G: 2.0582
Epoch [8/30]  Loss_D: 1.2370  Loss_G: 1.1362
Epoch [9/30]  Loss_D: 1.0485  Loss_G: 1.8982
Epoch [10/30]  Loss_D: 0.7779  Loss_G: 2.1802
Epoch [11/30]  Loss_D: 1.0948  Loss_G: 1.1336
Epoch [12/30]  Loss_D: 0.9221  Loss_G: 2.5415
Epoch [13/30]  Loss_D: 0.8969  Loss_G: 1.9823
Epoch [14/30]  Loss_D: 0.7232  Loss_G: 2.2849
Epoch [15/30]  Loss_D: 0.9782  Loss_G: 1.9287
Epoch [16/30]  Loss_D: 0.8561  Loss_G: 2.3729
Epoch [17/30]  Loss_D: 1.0216  Loss_G: 2.9472
Epoch [18/30]  Loss_D: 1.0415  Loss_G: 2.1573
Epoch [19/30]  Loss_D: 1.2468  Loss_G: 1.2154
Epoch [20/30]  Loss_D: 0.9167  Loss_G: 2.6390
Epoch [21/30]  Loss_D: 0.9084  Loss_G: 1.3701
Epoch [22/30]  Loss_D: 0.9902  Loss_G: 2.72

**NOTE**

GAN training can be understood as a continuous competition between the Generator and the Discriminator. The Discriminator learns to correctly distinguish between real and fake images, while the Generator simultaneously tries to produce realistic images to fool the Discriminator. The fluctuating loss values observed across epochs reflect this ongoing back-and-forth battle, where neither model fully dominates. This lack of stable convergence is a characteristic feature of GANs and indicates dynamic adversarial learning rather than traditional optimization.


### Final Output


In [10]:

with torch.no_grad():
    test_noise = torch.randn(10, nz, 1, 1, device=device)
    generated = netG(test_noise)
    save_image((generated + 1) / 2, "final_output.png", nrow=5)

print("✅ Training complete! Check generated images.")

✅ Training complete! Check generated images.


### Limitations
CIFAR-10 is a widely used dataset for image classification, but it has several limitations:

* Low resolution (32×32) → poor detail
* Small objects → hard to learn structure
* High class variation → complex learning
* RGB images → higher complexity
* Limited features → blurry outputs

Overall, while CIFAR-10 is useful for learning and experimentation, its limitations make it less suitable for high-quality image generation or real-world applications requiring detailed visuals.

